# 09 — Tìm điểm bắt đầu, kết thúc sóng T và vẽ quỹ đạo 3D

## 1. Notebook này làm gì?

Notebook tìm **hai mốc đầu và cuối của sóng T**, rồi lấy đoạn tín hiệu giữa hai mốc để vẽ trong không gian 3 chiều.

Các khái niệm dùng xuyên suốt:

- **QRS**: phần tín hiệu gắn với lúc các buồng thất được hoạt hoá điện.
- **Sóng T**: phần tín hiệu gắn với quá trình hồi phục điện của các buồng thất. **T-on** là điểm bắt đầu; **T-off** hay **T-end** là điểm kết thúc.
- **XYZ / VCG**: ba thành phần điện tim theo ba hướng không gian. Mỗi thời điểm có một bộ số `(X, Y, Z)`, tương ứng một điểm 3D.
- **T loop**: đường nối các điểm XYZ theo thời gian, từ T-on đến T-off. Đường này không nhất thiết khép kín hoàn toàn.
- **V1**: kết quả của cách tách cũ. **V2**: kết quả của cách Wavelet/TLS đang thử trong notebook này.
- **Wavelet**: phép biến đổi giúp làm rõ tín hiệu thay đổi ở những vị trí nào. **TLS**: cách tìm một đường thẳng đại diện cho một đám điểm 3D, từ đó chọn hướng quan sát. Hai bước này được giải thích ở mục 6 và 10.

**Luồng thực hiện:** lấy nhịp đại diện đã lưu → biến đổi từng trục → kết hợp ba trục → tìm hai mốc T → cắt và vẽ T loop.

Cách làm dựa trên [Martínez 2004](https://doi.org/10.1109/TBME.2003.821031) và [Almeida 2009](https://doi.org/10.1109/TBME.2009.2021658), có điều chỉnh cho dữ liệu của dự án. Chưa có đối chiếu đầu ra với chương trình của tác giả để khẳng định hai bên cho kết quả tương đương. Các công thức và điều chỉnh được ghi trong [tài liệu phương pháp](METHOD_REPRODUCTION.md).

## Chuẩn bị dữ liệu trên máy này
Chạy cell dưới một lần mỗi phiên. Dữ liệu PTB được tải từ GitHub Releases nếu còn thiếu.
Notebook 06–09 tự tạo các đầu vào phụ thuộc còn thiếu; lần đầu có thể chạy lâu.
Hướng dẫn và dung lượng: xem `README.md` và `docs/REPRODUCIBILITY.md`.

In [ ]:
from project_setup import prepare_notebook
prepare_notebook(9)

## 2. Chuẩn bị các hàm và thông số

**Mục đích:** nạp công cụ cần dùng và đọc kết quả đã có của V1.

Ba nhóm file chính:

- `vcg_t_almeida.py`: thực hiện phép biến đổi và tìm hai mốc T.
- `scripts/run_vcg_t_almeida_experiment.py`: đọc dữ liệu, lưu kết quả từng bản ghi và tổng hợp báo cáo.
- Các file `vcg_t_almeida_plots.py` và `vcg_t_almeida_gallery.py`: vẽ hình trong notebook và giao diện xem từng bản ghi.

Dữ liệu đầu vào là **nhịp đại diện đã lưu**, nằm trong `outputs/vcg_full_evaluation/artifacts/`. Cách tạo nhịp đại diện được giải thích ở mục 4.

**Cách đọc kết quả cell dưới:**

- `CFG` chứa các **thông số cài đặt**, như ngưỡng tìm sóng và số lần lặp tối đa. Đây chưa phải kết quả đo của bệnh nhân.
- `Implementation class` cho biết đây là bản dựng theo bài báo có điều chỉnh.
- `protected files` là số file V1 được theo dõi bằng **mã kiểm tra nội dung** (hash). Nếu nội dung file thay đổi, mã kiểm tra cũng thay đổi.

Các thông số về ngưỡng được giải thích ở mục 7; điều kiện dừng ở mục 13.

In [ ]:
from pathlib import Path
from dataclasses import asdict
import os, sys, json, subprocess
os.environ.setdefault('MPLCONFIGDIR', str(__import__('pathlib').Path(__import__('tempfile').gettempdir())/'vcg_t_almeida_mpl'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, IFrame

ROOT = Path.cwd().resolve()
if not (ROOT / 'vcg_t_almeida.py').exists():
    raise RuntimeError('Chạy notebook từ thư mục gốc project')
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / 'scripts'))
from vcg_t_almeida import (AlmeidaConfig, METHOD_CLASS, wavelet_transform,
    build_t_search_window, single_lead_initialization, select_multilead_scale,
    tls_direction, iterate_tls_boundary, project_wavelets, delineate_t, validate_t_bounds)
from vcg_pipeline import FEATURE_NAMES, cut_loop
from run_vcg_t_almeida_experiment import (initialize, process_one, run_experiment,
    verify_v1, DEFAULT_OUT, ARTIFACT_INPUT, safe_id, write_json)
from vcg_reference_metrics import load_median
from vcg_t_almeida_plots import (plot_median, plot_wavelets, plot_single_leads,
    plot_wt_loop, plot_tls_fit, plot_derived, plot_iterations, plot_t_loops)

# The batch renderer uses Agg; register the notebook PNG formatter AFTER
# importing it so display(fig) embeds actual figures, not text reprs.
get_ipython().run_line_magic('matplotlib', 'inline')
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('png')

CFG = AlmeidaConfig()  # paper/reference choices; not tuned on test
OUT = DEFAULT_OUT
frame, manifest, V1_CFG, protected, experiment_config = initialize(OUT, CFG)
print('Implementation class:', METHOD_CLASS)
print('Notebook08 thực sự có:', experiment_config['notebook08_available'])
print('V1 protected files:', len(protected))
display(pd.Series(asdict(CFG), name='value').to_frame())

## 3. Chọn bản ghi để xem từng bước

**Bản ghi** (recording) là một lần ghi tín hiệu, có mã riêng như `patient104/s0306lre`. Một người có thể có nhiều bản ghi.

Cell dưới mặc định dùng `patient104/s0306lre`. Đổi `RECORD_ID` để xem bản ghi khác, rồi chạy các cell phía sau theo thứ tự.

Các tên biến cần biết:

- `xyz_median`: tín hiệu XYZ của nhịp đại diện.
- `fs`: số điểm đo trong một giây, đơn vị Hz. Với `fs = 1000`, hai điểm liên tiếp cách nhau 1 mili giây (ms). Một điểm đo còn được gọi là **mẫu** (sample).
- `anchor`: vị trí mốc QRS dùng làm thời điểm **0 ms**.
- `qrs_on`, `qrs_off`: vị trí đầu và cuối đoạn QRS đã có từ V1.
- `RR`: khoảng thời gian giữa hai mốc nhịp liên tiếp.

Chương trình tìm T mới từ XYZ, thời gian, QRS và RR. Các mốc T cũ được đọc để đối chiếu sau khi đã tính V2.

**Cách đọc kết quả:** cell in mã bản ghi, trạng thái tìm T và lý do chưa tìm được nếu có. Các trạng thái `detected`, `needs_review`, `unresolved` được giải thích ở mục 14. Nếu không có nhịp đại diện, các bước sau sẽ báo thiếu dữ liệu.

In [ ]:
RECORD_ID = 'patient104/s0306lre'
original_row = frame.set_index('record_id').loc[RECORD_ID].to_dict()
original_row['record_id'] = RECORD_ID
result_row, diagnostic, cached = process_one(original_row, OUT, CFG, V1_CFG,
    experiment_config['algorithm_fingerprint'], resume=True)
median = load_median(ARTIFACT_INPUT, RECORD_ID)
with np.load(OUT / 'artifacts' / (safe_id(RECORD_ID)+'.npz'), allow_pickle=False) as saved:
    arrays = {k: saved[k].copy() for k in saved.files}
case = dict(row=result_row, median=median, diag=diagnostic, arrays=arrays)
xyz_median, fs, anchor = median['xyz'], median['fs'], median['anchor']
b = median['boundaries'] or {}
available = xyz_median is not None
print(RECORD_ID, result_row['status'], result_row['failure_reason'])
print('Source JSON:', median['json_path'])
print('Source NPZ SHA256:', median['signal_sha256'])

## 4. Nhịp đại diện được tạo bằng trung vị là gì?

**Trung vị** (median) là giá trị ở giữa sau khi sắp xếp một nhóm số. Ví dụ, trung vị của `1, 2, 9` là `2`.

Trong dự án này, các nhịp được căn theo mốc QRS. Tại **mỗi thời điểm tương ứng**, chương trình lấy trung vị của các giá trị X; làm tương tự với Y và Z. Nối các giá trị đó lại ta được **nhịp đại diện**. Chương trình không chọn nguyên một nhịp nằm ở giữa bản ghi.

Nhịp đại diện này đã qua các bước của V1: lọc tín hiệu, xử lý nhiễu điện lưới khi cần và đưa đoạn nền trước QRS, gọi là **đoạn PQ**, về gần mức 0. Vì vậy, chữ “gốc” trong các hình có nghĩa là mảng XYZ V1 đã lưu.

### Trung vị có thể làm mất đặc điểm thật không?

**Có thể.** Ví dụ minh hoạ: ở cùng một thời điểm, năm nhịp có giá trị `0, 0, 0, 0.8, 0.8 mV`; trung vị là `0 mV`. Nếu mức `0.8` là một đặc điểm thật chỉ có ở hai nhịp, đặc điểm đó có thể không còn trên nhịp đại diện.

Mã hiện tại giữ các nhịp có hình dạng khác và gắn cờ xem lại, nhưng bước gộp bằng trung vị vẫn có thể làm yếu đặc điểm của thiểu số nhịp. Trước đó, một số nhịp còn bị loại theo khoảng RR hoặc điều kiện dữ liệu. **Chưa có phân tích chứng minh các bước này giữ đủ thông tin phân biệt MI và HC.**

Do đó, kết quả dưới đây mô tả **nhịp đại diện**. Để đánh giá thông tin bị mất, cần đối chiếu với từng nhịp riêng. Đây cũng là vấn đề cần chú ý khi trộn nhiều dạng nhịp, như ví dụ trong [nghiên cứu BRAVEHEART](https://doi.org/10.1016/j.cmpb.2023.107798).

**Cách đọc hình:** ba đường là X, Y, Z; trục đứng là điện thế tính bằng mV, trục ngang là thời gian so với mốc QRS. Công thức đổi chỉ số thành thời gian là `t_ms = (chỉ_số_mẫu − anchor) × 1000 / fs`. Ví dụ ở 1000 Hz, mẫu nằm sau anchor 100 vị trí có thời gian +100 ms.

In [ ]:
if available:
    t = median['time_ms']
    assert xyz_median.shape == (len(t), 3) and np.isfinite(xyz_median).all()
    assert np.array_equal(t, (np.arange(len(t))-anchor)*1000/fs)
    print('shape=', xyz_median.shape, 'fs=', fs, 'anchor=', anchor,
          'QRS samples=', (b['qrs_on'],b['qrs_off']), 'PQ slice=',median['artifact']['pq_indices'])
fig = plot_median(case, comparison=False); display(fig); plt.close(fig)

## 5. Xác định khoảng thời gian được phép tìm sóng T

**Mục đích:** giới hạn vùng tìm kiếm để chương trình tập trung vào phần tín hiệu sau QRS.

Khoảng tìm T bắt đầu sau QRS-off và ít nhất 100 ms sau mốc QRS. Điểm kết thúc phụ thuộc khoảng RR, khoảng bảo vệ trước QRS tiếp theo và độ dài nhịp đại diện thực sự có sẵn.

Trong bản thử này, RR được thay bằng trung vị các khoảng RR của bản ghi. Đây là một điều chỉnh so với cách cập nhật RR theo từng nhịp trong mã tham khảo.

Cell cũng in một khoảng dùng tính **RMS**. RMS là cách đo mức biên độ chung: bình phương các giá trị, lấy trung bình rồi lấy căn bậc hai. Giá trị này giúp đặt ngưỡng phát hiện sóng ở mục 7.

**Cách đọc kết quả:**

- `T search samples`: chỉ số đầu/cuối của khoảng được tìm T; phần `ms` là cùng khoảng đó đổi sang mili giây.
- `RMS samples`: phần dữ liệu dùng tính RMS.
- `complete RR = False`: nhịp đại diện ngắn nên không chứa trọn khoảng RR cần có; chương trình dùng phần dữ liệu hiện hữu và lưu hạn chế này.

Các mốc T của V1 không tham gia đặt khoảng tìm kiếm mới.

In [ ]:
if available:
    rr = float(np.median(median['artifact']['rr_seconds']))
    search_window = build_t_search_window(len(xyz_median),fs,anchor,b['qrs_off'],rr,CFG)
    rms_interval = (anchor,min(len(xyz_median)-1,anchor+int(np.floor(rr*fs+.5))))
    print('RR=',rr,'s; T search samples=',search_window,'; ms=',t[list(search_window)])
    print('RMS samples=',rms_interval,'; complete RR=',diagnostic['rms_interval_complete_rr'])
else:
    print('Không có median: giữ trạng thái lỗi; không dựng search window giả.')

## 6. Dùng wavelet để nhìn rõ sự thay đổi của tín hiệu

**Mục đích:** tạo một cách biểu diễn giúp nhận ra các sườn lên/xuống của sóng T trên từng trục X, Y, Z.

Phép biến đổi **wavelet**, viết tắt **WT**, tạo ra các dãy giá trị mới từ XYZ. Những điểm cao hoặc thấp cục bộ của dãy mới được gọi là **cực trị**; chúng giúp nhận ra các sườn của sóng. Vị trí dãy WT đổi dấu qua 0 có thể hỗ trợ tìm đỉnh hoặc đáy của sóng T.

Chương trình tính ở ba **mức quan sát** (scale): 3, 4 và 5. Mức lớn hơn xét sự thay đổi trên phạm vi thời gian rộng hơn. Các số này là mức của bộ lọc trong phương pháp, không phải độ dài cửa sổ tính bằng ms.

Loại wavelet dùng ở đây có tên *quadratic spline*. Cách tính giữ số điểm ở mọi mức bằng số điểm của XYZ, nên có thể đối chiếu trực tiếp vị trí thời gian.

**Cách đọc hình 3 × 3 bên dưới:** mỗi hàng là một trục; mỗi cột là một mức. Các dấu chấm/dấu x cho biết cực trị được chọn hoặc bỏ; các đường ngang thể hiện ngưỡng. Các giá trị trên trục đứng là hệ số WT.

**Cách đọc bảng bộ lọc:** `taps` là số hệ số của bộ lọc; `delay_samples` là độ trễ cần bù để giữ đúng thời gian; `support_radius` cho biết phạm vi các mẫu lân cận ảnh hưởng tới kết quả. Bộ lọc được điều chỉnh theo `fs`; chi tiết công thức nằm trong [tài liệu phương pháp](METHOD_REPRODUCTION.md).

In [ ]:
if available:
    wavelets, filter_bank = wavelet_transform(xyz_median,fs,CFG)
    for scale in (3,4,5):
        assert np.array_equal(wavelets[scale], arrays['wavelet_scale'+str(scale)])
    bank_table = pd.DataFrame([dict(scale=k, taps=len(v['coefficients']),
        delay_samples=v['delay_samples'],alignment=v['alignment'],
        support_radius=v['support_radius_samples'],fs_adaptation=v['fs_adaptation'])
        for k,v in filter_bank.items()])
    display(bank_table)
    print('h=',filter_bank['4']['h'],'g=',filter_bank['4']['g'])
fig=plot_wavelets(case); display(fig); plt.close(fig)

## 7. Tìm sóng T thử trên từng trục X, Y, Z

**Mục đích:** lấy dấu hiệu ban đầu từ từng trục trước khi kết hợp ba trục. Trong mã, bước này được gọi là `single-lead` hoặc `SL`, nghĩa là xử lý riêng từng trục.

Mỗi trục bắt đầu ở mức 4. Nếu chưa phát hiện được sóng T thì thử mức 5. Chương trình dùng ba loại ngưỡng với ba việc khác nhau:

1. **Phát hiện có sóng:** dùng `0.25 × RMS(WT)` để kiểm tra các sườn có đủ rõ.
2. **Chọn các sườn đáng kể:** dùng `0.125 × giá trị tuyệt đối lớn nhất của WT` trong vùng tìm. Giá trị tuyệt đối là độ lớn sau khi bỏ dấu âm.
3. **Tìm đầu/cuối sóng:** dò ra phía ngoài các sườn đã chọn, dùng tỷ lệ biên độ để tìm nơi sóng bắt đầu hoặc kết thúc. Bước riêng từng trục dùng tỷ lệ đầu 0.25 và cuối 0.4.

Ví dụ, nếu sườn đầu có độ lớn 0.8 thì ngưỡng đầu theo tỷ lệ là `0.25 × 0.8 = 0.2`. Đây là ví dụ tính ngưỡng, chưa phải kết quả của bản ghi.

**Cách đọc bảng:**

- `wave_detected`: đã nhận ra một sóng T ứng viên trên trục đó hay chưa.
- `candidate_sample`: vị trí mẫu đang được đề xuất; chưa phải kết quả cuối.
- `morphology_type`: dạng sóng, như dương, âm hoặc hai pha (có cả phần dương và âm).
- `only_upwards/downwards`: chỉ tìm được một sườn, chưa đủ cấu trúc để xác định một sóng T đầy đủ.
- `failure_reason`: lý do bước đó chưa có kết quả.

Hình dưới cho thấy các mốc thử của từng trục. Chương trình có thể nhận ra sóng T nhưng vẫn chưa tìm đủ cả hai biên.

In [ ]:
if available:
    single_leads=single_lead_initialization(wavelets,search_window,fs,rms_interval,CFG)
    sl_columns=['axis','wavelet_scale','wave_detected','t_peak_candidate_sample',
      't_on_candidate_sample','t_off_candidate_sample','first_significant_extremum',
      'last_significant_extremum','morphology_type','detection_status','failure_reason']
    display(pd.DataFrame(single_leads)[sl_columns])
    display(pd.DataFrame([dict(axis=s['axis'],scale=a['wavelet_scale'],**e)
        for s in single_leads for a in s['attempts'] for e in a['extrema']]))
fig=plot_single_leads(case); display(fig); plt.close(fig)

## 8. Ghép ba trục wavelet thành một đường đi 3D

Ở cùng một thời điểm, ta có ba hệ số `Wx, Wy, Wz`. Coi bộ ba này là một điểm 3D rồi nối theo thứ tự thời gian, ta được **đường đi trong không gian wavelet**, gọi là *Wavelet Transform Loop*.

Đường này được dùng để tìm hướng kết hợp ba trục. T loop trên điện thế XYZ sẽ được vẽ ở mục 16.

**Cách đọc hình:** các trục `Wx, Wy, Wz` là ba hệ số wavelet. Đường xám biểu diễn phần nằm trong khoảng tìm T; các đoạn màu đánh dấu phần dữ liệu được các nhánh tìm đầu/cuối dùng để tìm hướng.

`WT loop shape = (N, 3)` nghĩa là có N thời điểm, mỗi thời điểm gồm ba hệ số. `ML scale` là mức dùng khi kết hợp nhiều trục; `ML` viết tắt của *multilead*.

In [ ]:
if available:
    ml_scale=select_multilead_scale(single_leads)
    W_xyz=wavelets[ml_scale]
    print('WT loop shape:',W_xyz.shape,'; ML scale=',ml_scale)
fig=plot_wt_loop(case);display(fig);plt.close(fig)

## 9. Chọn đoạn dữ liệu ban đầu để tìm hướng

**Mục đích:** xác định những điểm 3D sẽ được dùng trong lần tìm hướng đầu tiên.

Cách chọn mức chung: dùng mức 5 nếu ít nhất hai trong ba trục phát hiện T ở mức 5; các trường hợp còn lại dùng mức 4.

Đoạn dữ liệu ban đầu bao trùm các khoảng tìm kiếm của những trục đã phát hiện T: lấy điểm bắt đầu sớm nhất và điểm kết thúc muộn nhất. Vì thí nghiệm dùng cùng QRS cho cả ba trục, các khoảng này có cùng giới hạn.

Trong phần TLS, **cửa sổ W** là một đoạn thời gian dùng để tìm hướng của đám điểm. Các ký hiệu `Wx, Wy, Wz` ở mục 8 lại là các giá trị wavelet. Cửa sổ tìm hướng W có thể được cập nhật ở các vòng sau; khoảng được phép tìm T đã nêu ở mục 5 được giữ riêng.

**Cách đọc kết quả:** `Initial TLS W samples` và phần `ms` cho biết đoạn ban đầu. Nếu cả ba trục đều không phát hiện được T thì xuất `INITIALIZATION_FAILED`, nghĩa là chưa có cơ sở để bắt đầu tìm hướng. Một trục đã phát hiện T nhưng thiếu một biên vẫn có thể hỗ trợ bước này.

In [ ]:
if available:
    valid_sl=[s for s in single_leads if s['wave_detected']]
    if valid_sl:
        initial_window=(min(s['search_start'] for s in valid_sl),max(s['search_end'] for s in valid_sl))
        print('Initial TLS W samples=',initial_window,'; ms=',t[list(initial_window)])
        initial_fit=tls_direction(W_xyz[initial_window[0]:initial_window[1]+1],cfg=CFG)
        display(pd.Series(initial_fit,name='initial affine TLS'))
    else:
        initial_window=None
        print('INITIALIZATION_FAILED: không có single-lead T detection.')

## 10. Tìm hướng quan sát để xác định đầu sóng T

**Ý tưởng:** ba trục có thể thể hiện sườn đầu T rõ khác nhau. Chương trình tìm một hướng trong không gian wavelet để gộp thông tin của cả ba.

**TLS** (*Total Least Squares*, bình phương tối thiểu toàn phần) chọn đường thẳng sao cho tổng bình phương khoảng cách vuông góc từ các điểm tới đường thẳng là nhỏ nhất. Có thể hình dung là đặt một đường thẳng đi qua đám điểm theo hướng đại diện của nó.

Đường thẳng được phép đi qua tâm đám điểm, không bị ép đi qua gốc tọa độ. Trong mã, phép tính đại số tên **SVD** được dùng để tìm đường thẳng đó.

Các đại lượng được in hoặc ghi trên hình:

- `u_on`: ba số mô tả hướng tìm đầu sóng T. Chúng được chuẩn hoá để độ dài hướng bằng 1.
- `center`: tâm đám điểm dùng khi tìm đường thẳng.
- `fit_quality` hoặc `energy fraction`: phần mức biến thiên của đám điểm nằm theo hướng đã chọn, từ 0 đến 1. Đây là số mô tả phép tìm hướng; nó không phải xác suất biên T đúng.
- `chosen iteration`: vòng lặp được chọn để lấy kết quả; cách lặp được giải thích ở mục 13.

**Cách đọc hình:** các chấm là điểm wavelet trong cửa sổ W; đường thẳng là kết quả TLS; mũi tên chỉ hướng. Nếu dữ liệu không cho một hướng xác định, nhánh này có thể chưa giải được.

In [ ]:
if available and initial_window:
    on_branch=iterate_tls_boundary(wavelets,ml_scale,initial_window,search_window,rms_interval,fs,'t_on',CFG)
    print('T-on chosen iteration:',on_branch['chosen_iteration'],'; stop:',on_branch['stop_reason'])
    print('u_on=',on_branch['direction'])
fig=plot_tls_fit(case,'t_on');display(fig);plt.close(fig)

## 11. Tìm hướng riêng cho cuối sóng T

Bước này làm tương tự mục 10 nhưng tập trung vào sườn cuối của sóng T.

Sườn đầu và sườn cuối có thể thể hiện rõ theo những hướng khác nhau. Vì vậy, chương trình có **hai nhánh tính độc lập**: nhánh T-on tìm đầu và nhánh T-off tìm cuối. Hai nhánh bắt đầu từ cùng cửa sổ, sau đó có thể cập nhật khác nhau.

**Cách đọc hình và kết quả:** `u_off` là hướng dùng cho cuối sóng; `chosen iteration` là vòng được chọn; `stop` là lý do dừng. So hình này với mục 10 để thấy hướng và vùng điểm có thay đổi hay không.

Các hướng được tìm từ dữ liệu của từng bản ghi; không có một bộ trọng số XYZ cố định dùng chung cho tất cả bản ghi.

In [ ]:
if available and initial_window:
    off_branch=iterate_tls_boundary(wavelets,ml_scale,initial_window,search_window,rms_interval,fs,'t_off',CFG)
    print('T-off chosen iteration:',off_branch['chosen_iteration'],'; stop:',off_branch['stop_reason'])
    print('u_off=',off_branch['direction'])
fig=plot_tls_fit(case,'t_off');display(fig);plt.close(fig)

## 12. Từ hướng 3D tạo một đường tín hiệu để tìm hai mốc

**Mục đích:** biến ba dãy wavelet thành một dãy duy nhất, dễ dò đầu hoặc cuối sóng hơn.

Chương trình nhân mỗi dãy với thành phần tương ứng của hướng `u`, rồi cộng lại:

`Wd = uX × Wx + uY × Wy + uZ × Wz`

`Wd` được gọi là **wavelet tổng hợp** (*derived wavelet*). Ví dụ, với hướng `(0.8, 0.6, 0)`, đường tổng hợp là `0.8 × Wx + 0.6 × Wy`. Mỗi nhánh T-on/T-off dùng hướng của chính nó.

Khi tìm đường thẳng, chương trình dùng các điểm đã trừ tâm. Khi tính Wd, chương trình chiếu các hệ số wavelet ban đầu theo công thức trên.

Cách tìm biên trên Wd:

- **T-on:** từ sườn đầu đáng kể, dò ngược thời gian.
- **T-off:** từ sườn cuối đáng kể, dò xuôi thời gian.
- Chọn vị trí gần sườn nhất đạt ngưỡng tương đối hoặc là một điểm thấp cục bộ của độ lớn Wd. Điểm thấp cục bộ là điểm thấp hơn các điểm lân cận.

Ở bước tổng hợp, cả ngưỡng đầu và cuối đều dùng tỷ lệ 0.25 của sườn liên quan. Với T-off, tỷ lệ này khác 0.4 ở bước riêng từng trục. Ngưỡng theo sườn cũng khác ngưỡng phát hiện `0.25 × RMS` ở mục 7.

**Cách đọc hình:** đường đậm là Wd; vùng tô là cửa sổ tìm hướng; chấm là sườn liên quan; đường đứng là mốc đang đề xuất; đường ngang là ngưỡng. Không tìm được vị trí đạt điều kiện thì giữ biên ở trạng thái thiếu.

In [ ]:
if available:
    for boundary in ('t_on','t_off'):
        branch=diagnostic.get('branches',{}).get(boundary,{})
        if branch.get('chosen_iteration'):
            e=branch['history'][branch['chosen_iteration']-1]
            derived=project_wavelets(wavelets[ml_scale],e['direction'])
            assert np.array_equal(derived,arrays[boundary+'_derived_iterations'][e['derived_index']])
            print(boundary,'u=',e['direction'],'threshold=',e.get('boundary_threshold'))
        fig=plot_derived(case,boundary);display(fig);plt.close(fig)

## 13. Lặp lại để cập nhật hướng và quyết định lúc dừng

Một vòng lặp gồm: **chọn đoạn dữ liệu → tìm hướng → tạo Wd → tìm mốc T mới**. Sau đó chương trình dùng mốc mới để chọn đoạn dữ liệu cho vòng tiếp theo.

Cách cập nhật cửa sổ tìm hướng:

- Nhánh đầu: từ **T-on vòng trước trừ 61.2 ms** đến vị trí sườn đầu đã chọn.
- Nhánh cuối: từ vị trí sườn cuối đã chọn đến **T-off vòng trước cộng 61.2 ms**.

Số 61.2 ms bằng `4 × 15.3 ms`. Giá trị 15.3 ms được đặt cố định theo phương pháp tham khảo, hiện trong cấu hình dưới tên `sigma_cse_ms`. Các cửa sổ luôn bị giới hạn bởi dữ liệu thực sự có.

Chương trình dừng trong các trường hợp chính:

1. **Mốc đã ổn định:** hai vòng liên tiếp khác nhau ít hơn 2 mẫu. Ở 1000 Hz, chênh 0 hoặc 1 ms đạt điều kiện này; chênh đúng 2 ms thì chưa đạt.
2. **Vòng mới không tìm được sườn phù hợp hoặc không tăng biên độ sườn liên quan:** giữ kết quả đã chấp nhận ở vòng trước.
3. **Đủ 20 vòng:** dừng theo giới hạn cài đặt và ghi chưa xác nhận ổn định.

**Cách đọc bảng:** `iteration` là số vòng; `window_start/end` là giới hạn đoạn tìm hướng; `candidate_sample` là mốc thử; `accepted` cho biết vòng có được nhận không; `stop_reason` là lý do dừng. Đồ thị phía trên theo dõi vị trí mốc, phía dưới theo dõi độ lớn sườn qua các vòng.

In [ ]:
history_rows=[]
for boundary,branch in diagnostic.get('branches',{}).items():
    for e in branch['history']:
        history_rows.append({k:v for k,v in e.items() if k not in ('wave',)})
display(pd.DataFrame(history_rows))
fig=plot_iterations(case);display(fig);plt.close(fig)

## 14. Kiểm tra kết quả cuối có đủ điều kiện để cắt T loop

Hai mốc cuối phải nằm trong dữ liệu và theo thứ tự **QRS-off < T-on < T-off**. Các đỉnh T liên quan cũng phải phù hợp với khoảng được chọn.

**Cách đọc ba trạng thái:**

- `detected`: có đủ hai biên hợp lệ và không có cờ xem lại của bộ kiểm tra hiện tại.
- `needs_review`: có đủ biên nhưng có điều kiện cần xem lại, như khoảng T quá ngắn/dài hoặc kết quả phụ thuộc vùng mép tín hiệu.
- `unresolved`: chưa có đủ kết quả hợp lệ để tạo T loop, chẳng hạn thiếu một biên hoặc thiếu nhịp đại diện.

`t_duration_ms` là thời lượng T, bằng `T-off − T-on`. Khoảng 60–400 ms trong cấu hình chỉ dùng gắn cờ xem lại; chương trình không tự kéo hoặc cắt mốc để ép vào khoảng này.

Một biên riêng lẻ vẫn có thể được lưu khi biên còn lại thiếu. Chỉ khi có đủ cả hai biên hợp lệ mới xuất T loop. Các mốc thử và lý do thất bại vẫn nằm trong dữ liệu chẩn đoán quá trình tính.

Các trạng thái trên mô tả kết quả của chương trình. Việc xác nhận biên đúng cần mốc tham chiếu độc lập.

In [ ]:
if available:
    independent_result=delineate_t(xyz_median,fs,anchor,b['qrs_on'],b['qrs_off'],rr,CFG)
    for key in ('t_on_sample','t_off_sample','status','failure_reason'):
        assert independent_result.get(key)==diagnostic.get(key)
    display(pd.Series({k:independent_result.get(k) for k in ['status','failure_reason',
        't_on_sample','t_off_sample','t_on_ms','t_off_ms','t_duration_ms','review_flags']}))
else:
    print('V2 unresolved vì V1 không có median; không thể phân đoạn trên dữ liệu không tồn tại.')

## 15. Đặt kết quả V1 và V2 cạnh nhau

**Mục đích:** xem cách mới chọn hai mốc khác cách cũ bao nhiêu. Hai bên dùng cùng nhịp đại diện, nên phép so sánh này chưa đánh giá phần thông tin có thể đã mất khi tạo median.

Bốn bản ghi bên dưới được chọn từ trước:

- `patient104/s0306lre`: có đủ hai biên để xem toàn bộ quy trình.
- `patient051/s0179lre`: V1 có khoảng T dài 17 ms. V2 tìm được T-on nhưng vẫn thiếu T-off, nên chưa có T loop V2.
- `patient034/s0109lre`: V1 thiếu T-on; V2 hiện cũng chưa có đủ cặp biên.
- `patient033/s0157lre`: bước trước chưa tạo được nhịp đại diện.

**Cách đọc bảng:** hậu tố `_v1_ms` là thời gian theo cách cũ; `_v2_ms` là theo cách mới. `NaN` hoặc ô trống nghĩa là chưa có giá trị.

Khi đọc hiệu **V2 − V1**, số dương nghĩa V2 chọn muộn hơn, số âm nghĩa V2 chọn sớm hơn. Đây là chênh lệch giữa hai cách. Để gọi là sai số, cần so mỗi cách với mốc tham chiếu đã được chấp nhận.

**Cách đọc hình:** các đường đứng chỉ biên; các vùng tô cho thấy phần tín hiệu thực sự được đưa vào khoảng T của từng cách.

In [ ]:
sample_ids=['patient104/s0306lre','patient051/s0179lre','patient034/s0109lre','patient033/s0157lre']
sample_results=[]
for rid in sample_ids:
    row=frame.set_index('record_id').loc[rid].to_dict();row['record_id']=rid
    out_row,_,_=process_one(row,OUT,CFG,V1_CFG,experiment_config['algorithm_fingerprint'],resume=True)
    sample_results.append(out_row)
sample_frame=pd.DataFrame(sample_results)
display(sample_frame[['record_id','t_on_v1_ms','t_on_v2_ms','t_off_v1_ms','t_off_v2_ms','status','failure_reason']])
fig=plot_median(case,comparison=True);display(fig);plt.close(fig)

## 16. Cắt đoạn T trên XYZ và quan sát quỹ đạo 3D

**Mục đích:** lấy các điểm XYZ từ T-on tới T-off, rồi nối chúng theo thời gian để tạo T loop.

Lệnh `cut_loop(xyz_median, on, off)` lấy **cả điểm đầu và điểm cuối**. Ví dụ, lấy từ mẫu 100 đến mẫu 200 có 101 điểm; ở 1000 Hz, khoảng thời gian giữa đầu và cuối là 100 ms.

Loop được cắt từ nhịp đại diện XYZ đã lưu ở V1. Tọa độ và mốc nền PQ được giữ để có thể so vị trí của hai loop.

**Cách đọc hình:**

- Hai hình 3D là T loop V1 và V2, dùng cùng gốc tọa độ, giới hạn trục và tỉ lệ.
- Chấm tròn chỉ điểm bắt đầu; dấu × chỉ điểm kết thúc; mũi tên chỉ chiều thời gian.
- Các hình **XY, XZ, YZ** là hình chiếu: lần lượt nhìn hai thành phần và bỏ thành phần còn lại, giúp xem phần đường đi được thêm hoặc bớt.
- Nếu thiếu một biên, ô V2 báo chưa có loop hợp lệ.

Trong phần kiểm tra của cell, QRS loop cũng được đối chiếu với khoảng cắt V1 đã lưu.

In [ ]:
if available:
    on,off=diagnostic.get('t_on_sample'),diagnostic.get('t_off_sample')
    t_loop_v2=cut_loop(xyz_median,on,off)
    if t_loop_v2 is not None:
        assert np.array_equal(t_loop_v2,arrays['t_loop_v2'])
        print('V2 loop shape=',t_loop_v2.shape,'duration=',(off-on)*1000/fs,'ms')
    else:
        assert 't_loop_v2' not in arrays
        print('Không có loop V2 hợp lệ; xem failure_reason.')
    if 'qrs_loop_v1_unchanged' in arrays:
        assert np.array_equal(arrays['qrs_loop_v1_unchanged'],xyz_median[b['qrs_on']:b['qrs_off']+1])
fig=plot_t_loops(case);display(fig);plt.close(fig)

## 17. Xem kết quả của toàn bộ 448 bản ghi

Đến đây, các bước minh hoạ được mở rộng thành bảng kết quả và giao diện xem **toàn bộ danh sách nghiên cứu**, đôi khi gọi là *cohort*. Danh sách hiện có 448 bản ghi MI/HC.

- **MI**: nhóm có chẩn đoán nhồi máu cơ tim trong dữ liệu.
- **HC**: nhóm đối chứng khoẻ mạnh.
- **Manifest**: bảng danh sách ID, bệnh nhân, nhãn và tập train/valid/test đã chia sẵn.

Giao diện xem nhiều bản ghi được gọi là **gallery**. Có thể tìm ID, đổi bản ghi, xem các mức wavelet, chọn vòng TLS và xoay hình T loop 3D. Ca thiếu biên hoặc thiếu nhịp đại diện vẫn có trong danh sách.

Chương trình lưu kết quả từng bản ghi thành **điểm lưu để chạy tiếp** (checkpoint). **Cache** là các kết quả đã lưu có thể dùng lại. **Resume** là tiếp tục lần chạy, chỉ dùng lại khi dữ liệu đầu vào và cấu hình khớp.

**Cách dùng cell:** mặc định đọc kết quả đã đầy đủ. `REGENERATE_COHORT=True` yêu cầu gọi lại bước tổng hợp; các điểm lưu hợp lệ vẫn có thể được dùng lại.

[Mở gallery đủ 448 ID](outputs/vcg_t_almeida_v2_experiment/gallery/index.html). Muốn xem các trang hình của tất cả ID đã nhúng sẵn, mở [Notebook 08](08_QRS_T_Loop_Reference_Error_Analysis.ipynb), **mục 14F**.

In [ ]:
REGENERATE_COHORT=False
results_path=OUT/'v2_t_boundaries.csv'
complete=results_path.exists() and set(pd.read_csv(results_path).record_id)==set(frame.record_id)
if REGENERATE_COHORT or not complete:
    run_experiment(OUT,resume=True,plots=True,gallery=True)
summary=json.loads((OUT/'execution_summary.json').read_text())
results=pd.read_csv(results_path)
assert set(results.record_id)==set(manifest.record_id) and results.record_id.is_unique
display(pd.Series(summary['input_counts'],name='actual V1 inputs'))
display(results.groupby(['label_name','status']).size().unstack(fill_value=0))
print('Gallery:',OUT/'gallery/index.html')
display(IFrame('outputs/vcg_t_almeida_v2_experiment/gallery/index.html',width='100%',height=780))

## 18. Đọc các bảng tổng hợp: đủ kết quả chưa, thay đổi bao nhiêu, đúng đến đâu?

Ba điều này được báo riêng:

1. **Mức bao phủ (coverage):** có bao nhiêu bản ghi tạo được kết quả. Ví dụ, số T loop chia cho tổng 448 bản ghi khác số T loop chia cho 443 bản ghi có nhịp đại diện. Các ca thất bại vẫn được đếm.
2. **Chênh lệch giữa hai cách (disagreement):** V1 và V2 đặt mốc khác nhau bao nhiêu.
3. **Độ chính xác theo tham chiếu (reference accuracy):** mỗi cách lệch bao nhiêu so với mốc do người đánh giá độc lập xác nhận trên đúng tín hiệu.

**QC** là các kiểm tra gắn cờ cần xem lại. Số cờ ít chưa chứng minh biên đúng hơn; hai cách cũng có thể dùng các quy tắc gắn cờ khác nhau.

### Cách đọc các bảng

- Bảng bao phủ: `recordings` là số bản ghi; `medians` là số có nhịp đại diện; `both_t_boundaries` là số có đủ hai biên. `split` chỉ tập train/valid/test.
- Bảng chênh biên: `n_paired` là số ca có đủ giá trị ở cả hai cách; `median_v2_minus_v1_ms` là trung vị chênh có dấu; `median_absolute_disagreement_ms` là trung vị độ lớn chênh sau khi bỏ dấu.
- Bảng sai số: `n_reference` là số mốc tham chiếu được chấp nhận. **MAE** là trung bình độ lớn sai lệch; **RMSE** là căn bậc hai của trung bình bình phương sai lệch, nhạy hơn với sai lệch lớn. Đơn vị ở đây là ms. Khi chưa có tham chiếu, các ô này để thiếu.
- Bảng thay đổi **feature**: feature là một con số mô tả hình dạng hoặc chuyển động của loop, như độ lớn vector hay tốc độ thay đổi XYZ. `median_absolute_difference` là trung vị độ lớn thay đổi của con số đó giữa hai cách.

Các feature QRS được giữ theo V1. Feature T dùng cùng công thức V1, với tốc độ XYZ tính trên cả nhịp rồi lấy đoạn T. Tăng hoặc giảm feature tự nó chưa chứng minh khả năng phân biệt MI/HC tốt hơn.

Cell dưới cũng có phần chạy các phép kiểm thử mã. Chúng kiểm tra phép tính, điều kiện dừng và việc giữ dữ liệu nhất quán; các tín hiệu mô phỏng trong kiểm thử chưa thay thế việc xác nhận biên trên dữ liệu thật.

In [ ]:
display(pd.read_csv(OUT/'reports/coverage_by_label_and_split.csv'))
display(pd.read_csv(OUT/'reports/boundary_disagreement_summary.csv'))
display(pd.read_csv(OUT/'reports/reference_error_summary.csv'))
display(pd.read_csv(OUT/'reports/feature_change_summary.csv'))
display(pd.read_csv(OUT/'diagnostics/representative_cases.csv'))
feature_comparison=pd.read_csv(OUT/'v1_v2_feature_comparison.csv')
q=feature_comparison[feature_comparison.feature.str.startswith('QRS_')]
assert q.signed_difference.dropna().eq(0).all()
print('Reference:',summary['reference'])

RUN_TESTS=True
if RUN_TESTS:
    checked=subprocess.run([sys.executable,'-m','unittest','discover','-s','tests',
        '-p','test_vcg_t_almeida.py','-v'],cwd=ROOT,text=True,capture_output=True)
    print(checked.stdout+checked.stderr)
    assert checked.returncode==0

## 19. Kết luận hiện tại và những điểm còn cần kiểm tra

Notebook cho thấy cách tìm biên T và vẽ loop trên **nhịp đại diện đã lưu**. Có được một loop nghĩa là chương trình đã tìm đủ biên theo các điều kiện cài đặt; việc đánh giá độ đúng và giá trị phân biệt bệnh cần thêm bằng chứng.

Những hạn chế chính:

- **Gộp nhịp bằng trung vị có thể làm yếu đặc điểm chỉ có ở ít nhịp.** Cần xem các nhịp riêng và mức thay đổi giữa các nhịp để đánh giá thông tin phân biệt MI/HC có bị mất hay không. So V1 với V2 trên cùng median chưa trả lời được câu hỏi này.
- Nhịp đại diện chỉ chứa một đoạn thời gian ngắn. Vì vậy, khoảng RR và mức RMS trong bản thử được tính từ dữ liệu hiện có; cách xử lý vùng mép và điều chỉnh bộ lọc cũng có thể ảnh hưởng biên.
- Chưa có đối chiếu đầu ra với chương trình của nhóm tác giả để xác nhận tương đương toàn bộ phương pháp.
- Chưa có mốc T tham chiếu độc lập được chấp nhận để kết luận V1 hay V2 chính xác hơn.

Một bước nghiên cứu phù hợp tiếp theo là đối chiếu **feature từ median** với **feature của từng nhịp và độ biến thiên giữa các nhịp**, trước hết trên train/valid. Nhịp có giá trị khác phần lớn các nhịp khác chưa đủ để kết luận là nhiễu, cũng chưa đủ để kết luận mang dấu hiệu MI.

### Mở các kết quả đã lưu

- [Gallery: tìm theo ID, xem từng vòng và T loop 3D](outputs/vcg_t_almeida_v2_experiment/gallery/index.html).
- [Notebook 08, mục 14F: hình của toàn bộ 448 ID](08_QRS_T_Loop_Reference_Error_Analysis.ipynb).
- [Báo cáo kết quả](outputs/vcg_t_almeida_v2_experiment/reports/experiment_report.md).
- [Nguồn, công thức và các điều chỉnh của phương pháp](METHOD_REPRODUCTION.md).

Để chạy lại toàn bộ thí nghiệm từ thư mục gốc project:

```bash
python scripts/run_vcg_t_almeida_experiment.py
```

Cell cuối kiểm tra mã nội dung các file V1 và hiển thị báo cáo chi tiết. Báo cáo này là tài liệu kỹ thuật; các mục phía trên giải thích các khái niệm cần để đọc nó.

In [ ]:
integrity=verify_v1(protected)
display(pd.Series(integrity))
assert integrity['changed_files']==0
display(Markdown((OUT/'reports/experiment_report.md').read_text()))
write_json(OUT/'reports/notebook09_integrity.json',integrity)